# Effect Sizes and Confidence Intervals (R2-3.3)

Recomputes the effect sizes and 95% confidence intervals added in the second revision for tests that previously reported only a test statistic or a p-value. The Spearman interval for cumulative views uses the Fisher z transformation, with a bootstrap check. Every bootstrap uses B = 10,000 resamples and every permutation test uses 10,000 permutations, each with seed 42 (Table S16). Resampling is done within each group unless stated otherwise.

1. unique-dyad comparison (Section 4.2): Fisher odds ratio and the permutation difference with its bootstrap CI;
2. daily cross-community ratio (Section 4.2): Mann-Whitney rank-biserial r, mean daily difference, and the Spearman correlation between daily volume and daily ratio;
3. matched-length windows of 7, 14 and 21 days around 27 August (Section 4.2);
4. engagement (Sections 4.2 and 4.3): Kruskal-Wallis omnibus for view counts and the Spearman correlation between cumulative views and cross-community edge count;
5. first-mover accounts in the Affan Kurniawan cascade (Section 4.2): Wilson CI and binomial test against the network base rate;
6. selection of the 38 paired communities (Section 4.4): rank-biserial r for size and pre-shift porosity.

Inputs (repository paths): `derived-data/hashed/temporal_edges_with_community_hashed.csv`, `derived-data/hashed/network_nodes_with_community_hashed.csv`, `derived-data/hashed/Data_with_community_hashed.csv`, `derived-data/hashed/actor_engagement_structural_merged_hashed.csv`, `derived-data/hashed/affan_community_propagation_fixed_hashed.csv`, `derived-data/community_porosity.csv`.

In [1]:
import numpy as np, pandas as pd
from scipy import stats
from scipy.stats.contingency import odds_ratio
from statsmodels.stats.proportion import proportion_confint

B, SEED = 10_000, 42
def ci(x): return np.percentile(x, [2.5, 97.5])
def rank_biserial(x, y):            # positive when x tends to exceed y
    U = stats.mannwhitneyu(x, y, alternative='two-sided')[0]
    return 2 * U / (len(x) * len(y)) - 1
def boot_two(x, y, f):              # resample each group separately
    rng = np.random.default_rng(SEED)
    return np.array([f(rng.choice(x, len(x)), rng.choice(y, len(y))) for _ in range(B)])

ev = pd.read_csv('temporal_edges_with_community_hashed.csv')            # derived-data/hashed/
ev['day'] = pd.to_datetime(ev['date_day'])
ev['cross'] = ev['is_cross_community'].astype(str).str.lower().eq('true').astype(int)
D26, D27, D28 = pd.Timestamp('2025-08-26'), pd.Timestamp('2025-08-27'), pd.Timestamp('2025-08-28')
pre_ev, post_ev = ev[ev['day'] <= D26], ev[ev['day'] >= D28]          # 27 August excluded
print(f'events: pre {len(pre_ev):,}, post {len(post_ev):,}')

events: pre 10,670, post 10,344


## 1. Unique-dyad comparison (Section 4.2)
Each source-target pair is counted once per period.

In [2]:
pre_d  = pre_ev.drop_duplicates(['source', 'target'])['cross'].to_numpy()
post_d = post_ev.drop_duplicates(['source', 'target'])['cross'].to_numpy()
print(f'pre {pre_d.sum():,}/{len(pre_d):,} = {100*pre_d.mean():.1f}%   post {post_d.sum():,}/{len(post_d):,} = {100*post_d.mean():.1f}%')

tab = np.array([[post_d.sum(), len(post_d) - post_d.sum()], [pre_d.sum(), len(pre_d) - pre_d.sum()]])
o = odds_ratio(tab); lo, hi = o.confidence_interval(0.95)
print(f'Fisher exact: OR = {o.statistic:.2f}, 95% CI [{lo:.2f}, {hi:.2f}], p = {stats.fisher_exact(tab).pvalue:.1e}')

diff = post_d.mean() - pre_d.mean()
rng = np.random.default_rng(SEED); pooled = np.r_[pre_d, post_d]; n0 = len(pre_d)
perm = np.array([(lambda z: z[n0:].mean() - z[:n0].mean())(rng.permutation(pooled)) for _ in range(B)])
p_perm = (np.sum(np.abs(perm) >= abs(diff)) + 1) / (B + 1)
lo, hi = ci(boot_two(post_d, pre_d, lambda a, b: a.mean() - b.mean()))
print(f'permutation: difference = {100*diff:.1f} percentage points, 95% CI [{100*lo:.1f}, {100*hi:.1f}], p = {p_perm:.4f}')

pre 965/6,671 = 14.5%   post 1,248/7,202 = 17.3%


Fisher exact: OR = 1.24, 95% CI [1.13, 1.36], p = 4.3e-06


permutation: difference = 2.9 percentage points, 95% CI [1.6, 4.1], p = 0.0001


## 2. Daily cross-community ratio (Section 4.2)
Each day is one observation (26 pre-shift days, 33 post-shift days).

In [3]:
daily = ev[ev['day'] != D27].groupby('day').agg(n=('cross', 'size'), c=('cross', 'sum'))
daily['ratio'] = daily['c'] / daily['n']
dpre, dpost = daily.loc[daily.index <= D26, 'ratio'].to_numpy(), daily.loc[daily.index >= D28, 'ratio'].to_numpy()
p_mw = stats.mannwhitneyu(dpost, dpre, alternative='two-sided').pvalue
lo, hi = ci(boot_two(dpost, dpre, rank_biserial))
print(f'days: pre {len(dpre)}, post {len(dpost)} | median post {np.median(dpost):.3f}, pre {np.median(dpre):.3f}')
print(f'Mann-Whitney: rank-biserial r = {rank_biserial(dpost, dpre):.3f}, 95% CI [{lo:.3f}, {hi:.3f}], two-sided p = {p_mw:.4f}')
lo, hi = ci(boot_two(dpost, dpre, lambda a, b: a.mean() - b.mean()))
print(f'mean daily difference = {dpost.mean() - dpre.mean():.3f}, 95% CI [{lo:.3f}, {hi:.3f}]')

rho, p_rho = stats.spearmanr(daily['n'], daily['ratio'])
rng = np.random.default_rng(SEED); n, r = daily['n'].to_numpy(), daily['ratio'].to_numpy()
boot = [stats.spearmanr(n[i], r[i])[0] for i in (rng.integers(0, len(n), len(n)) for _ in range(B))]
lo, hi = ci(boot)
print(f'daily volume vs daily ratio ({len(n)} days): Spearman rho = {rho:.3f}, 95% CI [{lo:.3f}, {hi:.3f}], p = {p_rho:.4f}')

days: pre 26, post 33 | median post 0.227, pre 0.153
Mann-Whitney: rank-biserial r = 0.552, 95% CI [0.298, 0.774], two-sided p = 0.0003


mean daily difference = 0.086, 95% CI [0.043, 0.130]


daily volume vs daily ratio (59 days): Spearman rho = -0.471, 95% CI [-0.676, -0.217], p = 0.0002


## 3. Matched-length windows around 27 August (Section 4.2)
Pre-shift window: the last N days up to 26 August; post-shift window: the first N days from 28 August. Event level, chi-square without continuity correction.

In [4]:
def cramers_v(a, b):
    t = [[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]]
    chi2, p, _, _ = stats.chi2_contingency(t, correction=False)
    return chi2, p, np.sqrt(chi2 / (len(a) + len(b)))
for N in (7, 14, 21):
    a = ev[(ev['day'] > D26 - pd.Timedelta(days=N)) & (ev['day'] <= D26)]['cross'].to_numpy()
    b = ev[(ev['day'] >= D28) & (ev['day'] < D28 + pd.Timedelta(days=N))]['cross'].to_numpy()
    chi2, p, v = cramers_v(a, b)
    lo, hi = ci(boot_two(a, b, lambda x, y: cramers_v(x, y)[2]))
    print(f'{N:>2} days: {100*a.mean():.1f}% -> {100*b.mean():.1f}%, chi2 = {chi2:.2f}, p = {p:.3f}, '
          f"Cramer's V = {v:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")

 7 days: 16.8% -> 14.9%, chi2 = 8.67, p = 0.003, Cramer's V = 0.025, 95% CI [0.009, 0.042]


14 days: 16.7% -> 16.0%, chi2 = 1.39, p = 0.239, Cramer's V = 0.009, 95% CI [0.000, 0.025]


21 days: 15.8% -> 16.4%, chi2 = 1.47, p = 0.225, Cramer's V = 0.009, 95% CI [0.001, 0.024]


## 4. Engagement (Sections 4.2 and 4.3)

In [5]:
tw = pd.read_csv('Data_with_community_hashed.csv', sep=';')                    # derived-data/hashed/
tw['view_count'] = pd.to_numeric(tw['view_count'], errors='coerce')
s = tw.dropna(subset=['narrative', 'view_count'])
labels = sorted(s['narrative'].unique())
def eps2(y, lab):
    H = stats.kruskal(*[y[lab == k] for k in labels])[0]; n = len(y)
    return H * (n + 1) / (n**2 - 1)
y, lab = s['view_count'].to_numpy(), s['narrative'].to_numpy()
H, p = stats.kruskal(*[y[lab == k] for k in labels])
rng = np.random.default_rng(SEED)                                     # resample tweets, groups kept by label
boot = [eps2(y[i], lab[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(B))]
lo, hi = ci(boot)
print(f'view counts, {len(labels)} narratives, n = {len(y):,}: Kruskal-Wallis H = {H:.2f}, df = {len(labels)-1}, '
      f'p = {p:.1e}, epsilon^2 = {eps2(y, lab):.3f}, 95% CI [{lo:.3f}, {hi:.3f}]')

acc = pd.read_csv('actor_engagement_structural_merged_hashed.csv')    # derived-data/hashed/
views = tw.groupby('username_clean')['view_count'].sum()
acc['views'] = acc['username'].map(views).fillna(0)
v = acc[acc['views'] > 0]
x, z = v['views'].to_numpy(), v['cross_comm_out'].to_numpy()
rho, p = stats.spearmanr(x, z)
se = 1 / np.sqrt(len(x) - 3)                                          # Fisher z interval (reported)
lo, hi = np.tanh(np.arctanh(rho) + np.array([-1.96, 1.96]) * se)
print(f'cumulative views vs cross-community edge count, n = {len(v):,}: Spearman rho = {rho:.3f}, 95% CI [{lo:.3f}, {hi:.3f}] (Fisher z), p = {p:.2e}')
rng = np.random.default_rng(SEED)                                     # bootstrap check
boot = [stats.spearmanr(x[i], z[i])[0] for i in (rng.integers(0, len(x), len(x)) for _ in range(B))]
lo, hi = ci(boot)
print(f'  bootstrap check: 95% CI [{lo:.3f}, {hi:.3f}]')

view counts, 7 narratives, n = 8,510: Kruskal-Wallis H = 370.29, df = 6, p = 6.8e-77, epsilon^2 = 0.044, 95% CI [0.036, 0.053]
cumulative views vs cross-community edge count, n = 6,365: Spearman rho = 0.232, 95% CI [0.209, 0.255] (Fisher z), p = 8.73e-79


  bootstrap check: 95% CI [0.210, 0.254]


## 5. First-mover accounts in the Affan Kurniawan cascade (Section 4.2)
The account that first posted the narrative in each community, tested against the 2.1% non-grassroots share of the network.

In [6]:
fm = pd.read_csv('affan_community_propagation_fixed_hashed.csv')       # derived-data/hashed/
k, n = int((fm['first_account_type'] != 'grassroot').sum()), len(fm)
lo, hi = proportion_confint(k, n, method='wilson')
print(fm['first_account_type'].value_counts().to_dict())
print(f'non-grassroots first-movers: {k}/{n} = {100*k/n:.0f}%, 95% Wilson CI [{100*lo:.1f}, {100*hi:.1f}], '
      f'binomial p = {stats.binomtest(k, n, 0.021).pvalue:.1e} (base rate 2.1%)')

{'grassroot': 86, 'media/news': 10, 'elite/political': 4}
non-grassroots first-movers: 14/100 = 14%, 95% Wilson CI [8.5, 22.1], binomial p = 2.6e-08 (base rate 2.1%)


## 6. Selection of the 38 paired communities (Section 4.4)
Communities active in both windows against all other communities (size) and against excluded communities with a pre-shift value (pre-shift porosity).

In [7]:
nodes = pd.read_csv('network_nodes_with_community_hashed.csv', sep=';')  # derived-data/hashed/
por = pd.read_csv('community_porosity.csv')                             # derived-data/
size = nodes.groupby('community').size()
both = set(por.groupby('community')['phase'].nunique().loc[lambda s: s == 2].index)
pre_por = por[por['phase'] == 'Pre-shift'].set_index('community')['porosity']
sel, exc = sorted(both), sorted(set(size.index) - both)
for name, a, b in [('size', size.reindex(sel).to_numpy(), size.reindex(exc).to_numpy()),
                   ('pre-shift porosity', pre_por.reindex(sel).to_numpy(), pre_por.reindex([c for c in exc if c in pre_por.index]).to_numpy())]:
    p = stats.mannwhitneyu(a, b, alternative='two-sided').pvalue
    lo, hi = ci(boot_two(a, b, rank_biserial))
    print(f'{name}: selected n = {len(a)}, excluded n = {len(b)}, Mann-Whitney p = {p:.1e}, '
          f'rank-biserial r = {rank_biserial(a, b):.3f}, 95% CI [{lo:.3f}, {hi:.3f}]')

size: selected n = 38, excluded n = 1629, Mann-Whitney p = 9.2e-39, rank-biserial r = 0.997, 95% CI [0.994, 1.000]


pre-shift porosity: selected n = 38, excluded n = 75, Mann-Whitney p = 6.3e-17, rank-biserial r = 0.806, 95% CI [0.671, 0.922]
